# 06 · 离线分数和闭环轨迹为什么会不一致？

本课把同一个重载 checkpoint 放入真实 `env.step` 循环，并在完全相同的 held-out 初始条件下运行专家、未训练模型和行为克隆模型。你会同时看到 action MSE、车辆访问过的状态、终止原因、共同时间窗口和真实轨迹。

**前置课：Unit 02 状态估计。** 本课仍明确标记 privileged truth 的范围：它用于隔离行为克隆的训练与闭环问题，不能代表视觉或状态估计已经解决。

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from ad_tutorial.imitation import *

## 1. 先确认你测的是什么

offline MSE 的输入来自已保存 expert episode；closed-loop 的每个下一输入来自上一动作推动后的车辆状态。后者可能离开专家数据的窄分布。比较条件固定地图、时长、decision repeat、test offset 和 seed；test offset 为 `±0.4m`，未参与训练或模型选择。

In [ ]:
configs = default_split_configs(horizon=60)
data = collect_demonstrations(configs)
fit = train_behavioral_cloning(data.select("train"), data.select("validation"), BCTrainingConfig(epochs=70))
save_checkpoint(ROOT / "artifacts" / "imitation" / "lesson06_checkpoint.pt", fit)
bc = load_checkpoint(ROOT / "artifacts" / "imitation" / "lesson06_checkpoint.pt")
untrained = make_untrained_policy(bc.normalizer, seed=7, hidden_dim=32)
print("offline held-out test:", offline_mse(bc.model, bc.normalizer, data.select("test")))

In [ ]:
results = evaluate_closed_loop(configs["test"], bc, untrained_policy=untrained)
report = summarize_closed_loop(results)
display(pd.DataFrame({name: {
    "failure_rate": value["failure_rate"],
    "mean_abs_lateral_error_m": value["mean_abs_lateral_error_m"],
    "mean_distance_traveled_m": value["mean_distance_traveled_m"],
} for name, value in report.items()}).T)

## 2. 看 covariate shift，而不是猜测

对每个 rollout 重新提取 `before_*` 特征。专家、untrained 和 BC 访问的 feature 均值/范围可能不同；这就是状态分布变化的可见证据。把它和最后的失败标志一起看，不要用一张 loss 曲线推断“模型学会了驾驶”。

In [ ]:
rows = []
for policy_name, episodes in results.items():
    for index, episode in enumerate(episodes):
        f = features_from_trace(episode.trace)
        rows.append({"policy": policy_name, "test_index": index,
                      "feature_mean_e_y": f[:, 0].mean(),
                      "feature_max_abs_e_y": np.abs(f[:, 0]).max(),
                      "steps": episode.metrics["steps"],
                      "outcome": episode.metrics["outcome"],
                      "failure_reason": episode.metrics["failure_reason"]})
display(pd.DataFrame(rows))

## 3. 共同窗口和失败解释

如果一个策略提前失败，它没有和跑满 horizon 的策略经历相同长度。先报告终止原因和 distance，再只在 `min(steps)` 的共同窗口比较 lateral error。`horizon` 代表时间截断，不能写成到达；`failure` 需要以 MetaDrive 的 crash/out_of_road 等标志为证据。

In [ ]:
common = paired_common_window(results)
display(pd.DataFrame([
    {"policy": policy, "test_index": row["test_index"],
     "common_steps": row["common_steps"], **row["policies"][policy]}
    for row in common["per_config"]
    for policy in results
]))
display(pd.DataFrame(common["aggregate"]).T)

## 4. 同一模型的 in-distribution 对照

这是一个可编辑的短实验：沿用刚刚重载的同一个 checkpoint，只把初始偏移改为训练覆盖过的 `±0.1m`，运行真实闭环。它和 held-out `±0.4m` 的结果分开标记；如果 in-distribution 更稳定，只说明状态覆盖不同，不能推出道路泛化。

In [ ]:
in_distribution = [replace(config, initial_lateral_offset_m=float(np.sign(config.initial_lateral_offset_m) * 0.1))
                   for config in configs["test"]]
in_distribution_results = evaluate_closed_loop(in_distribution, bc)
display(pd.DataFrame([
    {"policy": name, "offsets": [-0.1, 0.1],
     "failure_rate": float(np.mean([episode.metrics["failure"] for episode in episodes])),
     "mean_steps": float(np.mean([episode.metrics["steps"] for episode in episodes]))}
    for name, episodes in in_distribution_results.items()
]))

## 5. 结论模板：把实际 CLI 观察写进去

完成 CLI 后，用下面的句式写结论，替换方括号中的实际值：

`命令 [完整命令] 在 [collection_time_s] 秒收集 [samples] 个样本，训练 [training_time_s] 秒；dataset hash=[hash]。held-out offline test MSE=[mse]。完整闭环 outcome/failure 为 expert=[...]/[...]、untrained=[...]/[...]、BC=[...]/[...]；paired common-window 的 BC mean lateral error=[...]、mean distance=[...]。最早失败原因为 [...]，对应 trace/GIF 是 [...]。因此本次结果支持 [...]，不支持 [...]；下一个受控实验是 [...]。`

只把 `horizon` 写成到达当 `arrive_dest=true`；如果是时间截断就写 `horizon`。比较时同时保留 full-outcome 和 paired common-window 两种数字。

## 6. 可编辑练习与答案

练习 A：把 test offset 改成 `±0.1m`，不要重训，比较 offline 与 closed-loop。你的预测是 covariate shift 变小还是仍然存在？

练习 B：把 `horizon` 加倍，记录哪一个指标最先改变，并写出它对应的轨迹证据。

答案：A 通常让状态更接近示范分布，但不能保证闭环一致；要以本次 rollout 的 feature 范围和失败标志判断。B 常见变化是 steps、distance 和 outcome；若模型较早漂移，延长 horizon 会暴露失败，而不是把之前的短时存活变成完成。不同机器或 MetaDrive 版本结果有差异，保留实际结果和配置。

选读 [DAgger](https://arxiv.org/abs/1011.0686)：bounded reading 只看 Introduction 和 algorithm idea。想象下一步实验：让学习者访问自己的状态，再让几何专家为这些状态给标签；写清楚新增数据来自哪里、哪些 test episode 仍必须保持 untouched。此处只讨论设计，没有把它冒充已完成的 DAgger 实验。